# 13.5B MoE 3 Expert, 1 Shared

In [1]:
## 📦 Environment Setup: Dependencies and Imports

import torch
import time
import os
import subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import gc
import sys
import importlib
from peft import PeftModel

from torch import nn
from torch.nn import functional as F

from copy import deepcopy

#!pip install -U bitsandbytes

# 🧠 Model Architecture: Llama-3-8B-UltraMedical

**Llama-3-8B-UltraMedical** is built on top of the LLaMA 3 architecture and fine-tuned on the UltraMedical dataset. In this notebook, we'll walk through its architecture and prepare it for Mixture-of-Experts (MoE) insertion.


In [3]:
# We need to apply the adapter permanently to the weights, to save this model.

# === Step 1: Define paths ===
model_name = "TsinghuaC3I/Llama-3-8B-UltraMedical"
peft_model_name = "xj2193/medmoe-orthopedic-expert"
save_path = "/content/merged-llama3-ortho-expert"

# === Step 2: Load base model with quant for memory efficiency (temporary) ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# === Step 3: Load PEFT adapter ===
model = PeftModel.from_pretrained(base_model, peft_model_name)

# === Step 4: Merge LoRA weights into base model
merged_model = model.merge_and_unload()

# === Step 5: Reload merged model in float16 ===
# Save temporarily to disk
temp_path = "/tmp/temp_merged_model"
merged_model.save_pretrained(temp_path)

# Now reload it in float16
merged_model_fp16 = AutoModelForCausalLM.from_pretrained(
    temp_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# === Step 6: Save the model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

merged_model_fp16.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Saved clean float16 model to: {save_path}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

✅ Saved clean float16 model to: /content/merged-llama3-ortho-expert


## 🔍 Step 1: Load and Inspect Model Layers

We begin by loading the model to examine its transformer block structure.


In [19]:
# Load Model
model_name = "/content/merged-llama3-ortho-expert"
ortho = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="cuda",
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [23]:
# 🔍 Inspect Transformer Layers After MoE Injection

transformer_layers = ortho.model.layers

for i, layer in enumerate(transformer_layers):
    layer_name = getattr(layer, 'name', f"LlamaDecoderLayer_{i}")
    total_params = sum(p.numel() for p in layer.parameters())
    print(f"Layer {i}: {layer_name} with {total_params:,} parameters")

print(f"Total parameters: {sum(p.numel() for p in ortho.parameters())/1e9:.1f}B")

Layer 0: LlamaDecoderLayer_0 with 109,060,096 parameters
Layer 1: LlamaDecoderLayer_1 with 109,060,096 parameters
Layer 2: LlamaDecoderLayer_2 with 109,060,096 parameters
Layer 3: LlamaDecoderLayer_3 with 109,060,096 parameters
Layer 4: LlamaDecoderLayer_4 with 109,060,096 parameters
Layer 5: LlamaDecoderLayer_5 with 109,060,096 parameters
Layer 6: LlamaDecoderLayer_6 with 109,060,096 parameters
Layer 7: LlamaDecoderLayer_7 with 109,060,096 parameters
Layer 8: LlamaDecoderLayer_8 with 109,060,096 parameters
Layer 9: LlamaDecoderLayer_9 with 109,060,096 parameters
Layer 10: LlamaDecoderLayer_10 with 109,060,096 parameters
Layer 11: LlamaDecoderLayer_11 with 109,060,096 parameters
Layer 12: LlamaDecoderLayer_12 with 109,060,096 parameters
Layer 13: LlamaDecoderLayer_13 with 109,060,096 parameters
Layer 14: LlamaDecoderLayer_14 with 109,060,096 parameters
Layer 15: LlamaDecoderLayer_15 with 109,060,096 parameters
Layer 16: LlamaDecoderLayer_16 with 109,060,096 parameters
Layer 17: LlamaDe

## Examine the transformer layers

In [3]:
# Peek at the transformer layers
transformer_layers = model.model.layers
print(f"Number of layers: {len(transformer_layers)}")
for idx,layer in enumerate(transformer_layers):
    print(f"Layer Index {idx}:\n {layer}")

Number of layers: 32
Layer Index 0:
 LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
    (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
    (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
    (act_fn): SiLU()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)
Layer Index 1:
 LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
    

# 🧩 MoE_LlamaMLP: Replacing the FFN with a Mixture of Experts

In this section, we define `MoE_LlamaMLP`, a drop-in replacement for the original `LlamaMLP`. It supports:

- 3 parallel FFN experts
- A lightweight Router MLP
-

The original `mlp` block looks like this:

```
LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
    (act_fn): SiLU()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)
```

### We will make modifcaitons

In [4]:
# === MLP Expert Block ===
class LlamaAttention(nn.Module):
    def __init__(self, gate_proj, up_proj, down_proj, act_fn):
        super().__init__()
        self.gate_proj = deepcopy(gate_proj)
        self.up_proj = deepcopy(up_proj)
        self.down_proj = deepcopy(down_proj)
        self.act_fn = act_fn

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))


# === MoE Router Block ===
class LlamaMLP_MoE(nn.Module):  # now supports expert freezing
    def __init__(self, hidden_size, expert_fn, num_experts=4, num_shared=1, train_expert_idx=None):
        super().__init__()
        self.num_experts = num_experts
        self.num_shared = num_shared

        # Router gate (learns distribution over non-shared experts only)
        self.router = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, num_experts)
        )

        # Non-shared experts (trainable)
        self.experts = nn.ModuleList([deepcopy(expert_fn) for _ in range(num_experts)])

        # Shared expert(s) (always participate)
        self.shared_experts = nn.ModuleList([deepcopy(expert_fn) for _ in range(num_shared)])

        # Store training index and freeze non-target experts if specified
        self.train_expert_idx = train_expert_idx
        if train_expert_idx is not None:
            for i, expert in enumerate(self.experts):
                if i != train_expert_idx:
                    for param in expert.parameters():
                        param.requires_grad = False

    def forward(self, x):
        B, T, H = x.shape

        # Compute routing weights for trainable experts
        gate_logits = self.router(x)                          # [B, T, num_experts]
        if self.train_expert_idx is not None:
            weights = torch.zeros_like(gate_logits)
            weights[..., self.train_expert_idx] = 1.0
        else:
            weights = torch.softmax(gate_logits, dim=-1)         # [B, T, num_experts]

        # Run all experts
        expert_outs = torch.stack([expert(x) for expert in self.experts], dim=-1)     # [B, T, H, num_experts]
        shared_outs = torch.stack([se(x) for se in self.shared_experts], dim=-1)      # [B, T, H, num_shared]

        # Weighted sum over learned experts
        routed = (weights.unsqueeze(2) * expert_outs).sum(-1)                         # [B, T, H]

        # Sum shared experts equally
        shared = shared_outs.mean(-1)                                                 # [B, T, H]

        return routed + shared


# === Inject into model ===
def inject_moe_into_llama(model, num_experts=4, num_shared=1, train_expert_idx=None):
    for i, layer in enumerate(model.model.layers):
        mlp = layer.mlp

        # Wrap original FFN block as expert template
        expert_template = LlamaAttention(
            mlp.gate_proj,
            mlp.up_proj,
            mlp.down_proj,
            mlp.act_fn
        )

        # Replace with MoE Router
        moe = LlamaMLP_MoE(
            hidden_size=mlp.hidden_size,
            expert_fn=expert_template,
            num_experts=num_experts,
            num_shared=num_shared,
            train_expert_idx=train_expert_idx
        )

        model.model.layers[i].mlp = moe.to(model.device).to(model.dtype)

    print(f"✅ Injected MoE into all {len(model.model.layers)} layers.")


In [5]:
# Surgery
inject_moe_into_llama(
    model,
    num_experts=3,
    num_shared=1
)

✅ Injected MoE into all 32 layers.


In [6]:
# Memory cleanup
gc.collect()
torch.cuda.empty_cache()

In [8]:
!nvidia-smi
#20.5GB

Sun May 11 07:08:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             53W /  400W |   21003MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [17]:
# Inspect new architecture
transformer_layers = model.model.layers
print(f"Number of layers: {len(transformer_layers)}")
print(f"New Architecture:\n {transformer_layers[0]}")

Number of layers: 32
New Architecture:
 LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP_MoE(
    (router): Sequential(
      (0): Linear(in_features=4096, out_features=4096, bias=True)
      (1): SiLU()
      (2): Linear(in_features=4096, out_features=3, bias=True)
    )
    (experts): ModuleList(
      (0-2): 3 x LlamaAttention(
        (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
    )
    (shared_experts): ModuleList(
      (0): LlamaAttention(
    

## New Structur


```
 LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP_MoE(
    (router): Sequential(
      (0): Linear(in_features=4096, out_features=4096, bias=True)
      (1): SiLU()
      (2): Linear(in_features=4096, out_features=2, bias=True)
    )
    (experts): ModuleList(
      (0-1): 2 x LlamaAttention(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
    )
    (shared_experts): ModuleList(
      (0): LlamaAttention(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
    )
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)
```



In [10]:
# 🔍 Inspect Transformer Layers After MoE Injection

transformer_layers = model.model.layers

for i, layer in enumerate(transformer_layers):
    layer_name = getattr(layer, 'name', f"LlamaDecoderLayer_{i}")
    total_params = sum(p.numel() for p in layer.parameters())
    print(f"Layer {i}: {layer_name} with {total_params:,} parameters")

print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e9:.1f}B")

Layer 0: LlamaDecoderLayer_0 with 390,094,851 parameters
Layer 1: LlamaDecoderLayer_1 with 390,094,851 parameters
Layer 2: LlamaDecoderLayer_2 with 390,094,851 parameters
Layer 3: LlamaDecoderLayer_3 with 390,094,851 parameters
Layer 4: LlamaDecoderLayer_4 with 390,094,851 parameters
Layer 5: LlamaDecoderLayer_5 with 390,094,851 parameters
Layer 6: LlamaDecoderLayer_6 with 390,094,851 parameters
Layer 7: LlamaDecoderLayer_7 with 390,094,851 parameters
Layer 8: LlamaDecoderLayer_8 with 390,094,851 parameters
Layer 9: LlamaDecoderLayer_9 with 390,094,851 parameters
Layer 10: LlamaDecoderLayer_10 with 390,094,851 parameters
Layer 11: LlamaDecoderLayer_11 with 390,094,851 parameters
Layer 12: LlamaDecoderLayer_12 with 390,094,851 parameters
Layer 13: LlamaDecoderLayer_13 with 390,094,851 parameters
Layer 14: LlamaDecoderLayer_14 with 390,094,851 parameters
Layer 15: LlamaDecoderLayer_15 with 390,094,851 parameters
Layer 16: LlamaDecoderLayer_16 with 390,094,851 parameters
Layer 17: LlamaDe

In [14]:
from transformers import AutoTokenizer

# Load tokenizer for your model
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          device_map="cuda",
                                          trust_remote_code=True,
                                          )

# Input prompt
prompt = "What is hypertension?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Forward pass through model
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    print("✅ Logits shape:", logits.shape)


✅ Logits shape: torch.Size([1, 4, 128256])


In [18]:
output = model.generate(**inputs)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is hypertension? The Journal of the American Medical Association, 2008; 300: 1021-102


In [16]:
print(output)

tensor([[ 3923,   374, 63308,    30,   578, 10139,   315,   279,  3778, 13235,
         10229,    11,   220,  1049,    23,    26,   220,  3101,    25,   220,
          4278,    16,    12,  4278]], device='cuda:0')


In [24]:
# Save our new MoE
from transformers import AutoTokenizer
import torch
import json
import os

# ✅ Paths
save_path = "/content/drive/MyDrive/medmoe/MoE/Llama3-UltraMedical-MoE-4x3.5B-14B"

# ✅ Save model weights (optionally use safetensors=True if needed)
model.save_pretrained(save_path)

# ✅ Save tokenizer
tokenizer.save_pretrained(save_path)

# ✅ Update config.json with correct metadata
# Update config
config_path = os.path.join(save_path, "config.json")

with open(config_path, "r") as f:
    config = json.load(f)

# Manual config patch
config.update({
    "architectures": ["LlamaForCausalLM"],
    "model_type": "llama",
    "torch_dtype": "float16",
    "use_cache": False,
    "moe_injected": True,
    "model_name": "Llama3-UltraMedical-MoE-4x3.5B-14B"
})

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)